In [7]:
!pip install backtesting

In [8]:
from backtesting import Backtest, Strategy
print("Installation OK")

Installation OK


In [9]:
import yfinance as yf

data = yf.download('AAPL', period='2y')
print(data.head())

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2024-08-08  211.478058  212.360414  207.036537  211.279779  47161100
2024-08-09  214.382904  214.918260  210.149571  210.278460  42201600
2024-08-12  215.911469  217.876732  213.995836  214.462340  38028100
2024-08-13  219.623627  220.239009  217.380433  217.380433  44155300
2024-08-14  220.070282  221.370532  218.065308  218.928845  41960600


In [10]:
import yfinance as yf

data = yf.download('AAPL', period='2y')

# Aplatir les colonnes à deux étages (garder seulement le 1er niveau : Open, High...)
data.columns = data.columns.get_level_values(0)

print(data.head())
print(data.columns) 
print(data.columns.tolist())    # ['Open', 'High', 'Low', 'Close', 'Volume']
print(data.isnull().sum())      # compter les valeurs manquantes

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Date                                                                
2024-08-08  211.478088  212.360444  207.036567  211.279809  47161100
2024-08-09  214.382935  214.918290  210.149601  210.278490  42201600
2024-08-12  215.911469  217.876732  213.995836  214.462340  38028100
2024-08-13  219.623642  220.239024  217.380448  217.380448  44155300
2024-08-14  220.070297  221.370548  218.065323  218.928860  41960600
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')
['Close', 'High', 'Low', 'Open', 'Volume']
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [11]:
import yfinance as yf
import pandas as pd
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

# 1. Données préparées
data = yf.download('AAPL', period='2y')
data.columns = data.columns.get_level_values(0)

# 2. La fonction SMA
def SMA(valeurs, n):
    return pd.Series(valeurs).rolling(n).mean()

# 3. La stratégie
class CroisementSMA(Strategy):
    n_courte = 10
    n_longue = 20

    def init(self):
        prix = self.data.Close
        self.sma_courte = self.I(SMA, prix, self.n_courte)
        self.sma_longue = self.I(SMA, prix, self.n_longue)

    def next(self):
        if crossover(self.sma_courte, self.sma_longue):
            self.buy()
        elif crossover(self.sma_longue, self.sma_courte):
            self.position.close()

# 4. Lancer
bt = Backtest(data, CroisementSMA, cash=10000, commission=0)
resultats = bt.run()
print(resultats)

[*********************100%***********************]  1 of 1 completed


Backtest.run:   0%|          | 0/481 [00:00<?, ?bar/s]

Start                     2024-08-08 00:00:00
End                       2026-08-07 00:00:00
Duration                    729 days 00:00:00
Exposure Time [%]                    59.48104
Equity Final [$]                  11023.58077
Equity Peak [$]                   11924.48043
Return [%]                           10.23581
Buy & Hold Return [%]                41.95469
Return (Ann.) [%]                     5.03418
Volatility (Ann.) [%]                19.95182
CAGR [%]                              5.00377
Sharpe Ratio                          0.25232
Sortino Ratio                         0.36296
Calmar Ratio                          0.21463
Alpha [%]                            -8.05487
Beta                                  0.43596
Max. Drawdown [%]                   -23.45529
Avg. Drawdown [%]                     -4.8525
Max. Drawdown Duration      503 days 00:00:00
Avg. Drawdown Duration       51 days 00:00:00
# Trades                                   12
Win Rate [%]                      

In [13]:
bt.plot(superimpose=False)

C:\Users\HP 1030 G7\anaconda3\Lib\site-packages\backtesting\_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
C:\Users\HP 1030 G7\anaconda3\Lib\site-packages\backtesting\_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(


GridPlot(id='p1805', ...)

In [18]:
# 1. Couper en deux (chronologiquement !)
n = len(data)
point_coupure = int(n * 0.7)        # 70% pour le train

data_train = data.iloc[:point_coupure]    # la partie ANCIENNE
data_test  = data.iloc[point_coupure:]    # la partie RÉCENTE

# Backtest sur le TRAIN
bt_train = Backtest(data_train, CroisementSMA, cash=10000, commission=0)

# Optimiser les paramètres SUR LE TRAIN
resultats_train = bt_train.optimize(
    n_courte=range(5, 30, 5),       # teste 5, 10, 15, 20, 25
    n_longue=range(30, 100, 10),    # teste 30, 40, 50, 60, 70, 80, 90
    maximize='Return [%]'
)

# Récupérer les meilleurs paramètres trouvés
meilleure_courte = resultats_train._strategy.n_courte
meilleure_longue = resultats_train._strategy.n_longue

print("Meilleurs paramètres sur le TRAIN :", meilleure_courte, meilleure_longue)
print("Return sur le TRAIN :", resultats_train['Return [%]'])

#Backtest sur le TEST, avec les paramètres du TRAIN
bt_test = Backtest(data_test, CroisementSMA, cash=10000, commission=0)
resultats_test = bt_test.run(
    n_courte=meilleure_courte,
    n_longue=meilleure_longue)

print("Return sur le TEST :", resultats_test['Return [%]'])
print("Buy & Hold sur le TEST :", resultats_test['Buy & Hold Return [%]'])


ValueError: attempt to get argmax of an empty sequence